# 07 — Phân Tích Tần Số Theo Từng Class
// Manual waveform & frequency analysis per class

**Mục tiêu**: Phân tích chi tiết waveform, spectrogram, và phân bố tần số cho từng class trong UrbanSound8K.

Giúp hiểu:
- Mỗi loại âm thanh nằm ở dải tần số nào?
- Tại sao chọn bandpass 50–10,000 Hz?
- Class nào dễ/khó phân biệt và tại sao?

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from IPython.display import Audio, display, HTML

import config
from src.data_loader import load_metadata, load_audio, get_file_path
from src.dsp_pipeline import pipeline_b_process

plt.style.use('dark_background')
%matplotlib inline

metadata = load_metadata()
sr = config.TARGET_SR
print(f"Dataset: {len(metadata)} clips, {metadata['class'].nunique()} classes")
print(f"Sample rate: {sr} Hz | Nyquist: {sr//2} Hz")

## 1. Khái Niệm Cơ Bản

### Tần số là gì?
- **Tần số (Hz)** = số lần dao động trong 1 giây
- Tai người nghe được: **20 Hz → 20,000 Hz**
- Sample rate 22,050 Hz → tần số cao nhất biểu diễn được: **11,025 Hz** (Nyquist)

### Các dải tần số
| Dải | Phạm vi | Âm thanh điển hình |
|-----|---------|--------------------|
| Sub-bass | 0–100 Hz | Rung nền, DC offset |
| Bass | 100–250 Hz | Tiếng máy, bass nhạc |
| Low-mid | 250–1,000 Hz | Giọng nói nam, còi xe, chó sủa |
| Mid | 1,000–4,000 Hz | Giọng trẻ em, siren, harmonic |
| High | 4,000–8,000 Hz | Hài bậc cao, drilling |
| Very high | 8,000–11,025 Hz | Sibilance, hiss |

### Cách xác định dải tần số
1. **FFT (Fast Fourier Transform)**: Chuyển tín hiệu từ miền thời gian → miền tần số
2. **Spectrogram**: Biểu đồ 2D (thời gian × tần số), màu = cường độ
3. **Mel Spectrogram**: Spectrogram theo thang Mel (gần tai người hơn)

In [ ]:
def analyze_class(cls_name, n_samples=3, show_raw_vs_dsp=True):
    """
    Phân tích chi tiết 1 class: waveform, FFT, spectrogram, mel spectrogram.
    
    Args:
        cls_name: Tên class (VD: 'dog_bark')
        n_samples: Số mẫu hiển thị
        show_raw_vs_dsp: Hiển thị so sánh raw vs DSP
    """
    subset = metadata[metadata['class'] == cls_name]
    foreground = subset[subset['salience'] == 1]
    if len(foreground) >= n_samples:
        samples = foreground.sample(n=n_samples, random_state=42)
    else:
        samples = subset.sample(n=min(n_samples, len(subset)), random_state=42)
    
    cls_id = config.CLASS_NAMES.index(cls_name)
    vn_names = {
        'air_conditioner': 'Máy điều hòa', 'car_horn': 'Còi xe hơi',
        'children_playing': 'Trẻ em chơi đùa', 'dog_bark': 'Chó sủa',
        'drilling': 'Khoan đường', 'engine_idling': 'Động cơ không tải',
        'gun_shot': 'Tiếng súng', 'jackhammer': 'Búa khoan',
        'siren': 'Còi báo động', 'street_music': 'Nhạc đường phố'
    }
    
    print(f"{'='*70}")
    print(f"  Class {cls_id}: {cls_name} ({vn_names[cls_name]})")
    print(f"  Tổng clip: {len(subset)} | Foreground: {len(foreground)} | Background: {len(subset)-len(foreground)}")
    print(f"{'='*70}")
    
    # Collect FFT data for energy analysis
    all_fft_mag = []
    
    for i, (_, row) in enumerate(samples.iterrows()):
        fpath = get_file_path(row)
        y_raw = load_audio(fpath)
        y_dsp = pipeline_b_process(y_raw)
        
        # Play audio
        print(f"\n--- Mẫu {i+1}: {row['slice_file_name']} (fold {row['fold']}, salience {row['salience']}) ---")
        display(Audio(y_raw, rate=sr))
        
        # ============ FIGURE: 4-panel analysis ============
        fig = plt.figure(figsize=(18, 12))
        fig.patch.set_facecolor('#0a0a0a')
        gs = gridspec.GridSpec(3, 2, hspace=0.35, wspace=0.25)
        
        # --- Panel 1: Waveform (Raw vs DSP) ---
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.set_facecolor('#111111')
        t = np.arange(len(y_raw)) / sr
        ax1.plot(t, y_raw, color='#888888', alpha=0.7, linewidth=0.5, label='Raw')
        ax1.plot(t, y_dsp, color='#00ff41', alpha=0.8, linewidth=0.5, label='DSP')
        ax1.set_title('Waveform (Dạng sóng)', color='#00ffff', fontsize=11)
        ax1.set_xlabel('Thời gian (s)', color='#888888')
        ax1.set_ylabel('Biên độ', color='#888888')
        ax1.legend(fontsize=8)
        ax1.grid(True, alpha=0.2)
        
        # --- Panel 2: FFT Spectrum ---
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.set_facecolor('#111111')
        
        # FFT of raw
        fft_raw = np.abs(np.fft.rfft(y_raw))
        fft_dsp = np.abs(np.fft.rfft(y_dsp))
        freqs = np.fft.rfftfreq(len(y_raw), 1/sr)
        all_fft_mag.append(fft_dsp)
        
        # Plot in dB
        fft_raw_db = 20 * np.log10(fft_raw + 1e-10)
        fft_dsp_db = 20 * np.log10(fft_dsp + 1e-10)
        ax2.plot(freqs, fft_raw_db, color='#888888', alpha=0.5, linewidth=0.5, label='Raw')
        ax2.plot(freqs, fft_dsp_db, color='#00ff41', alpha=0.8, linewidth=0.5, label='DSP')
        
        # Mark filter boundaries
        ax2.axvline(config.FILTER_LOW_FREQ, color='#ff0080', linestyle='--', alpha=0.7, label=f'Bandpass {config.FILTER_LOW_FREQ} Hz')
        ax2.axvline(config.FILTER_HIGH_FREQ, color='#ff0080', linestyle='--', alpha=0.7, label=f'Bandpass {config.FILTER_HIGH_FREQ} Hz')
        
        # Mark peak frequency
        peak_idx = np.argmax(fft_dsp[1:]) + 1  # skip DC
        peak_freq = freqs[peak_idx]
        ax2.axvline(peak_freq, color='#ffaa00', linestyle=':', alpha=0.8)
        ax2.annotate(f'Peak: {peak_freq:.0f} Hz', xy=(peak_freq, fft_dsp_db[peak_idx]),
                    xytext=(peak_freq + 500, fft_dsp_db[peak_idx] + 5),
                    color='#ffaa00', fontsize=8, arrowprops=dict(arrowstyle='->', color='#ffaa00'))
        
        ax2.set_title('FFT Spectrum (Phổ tần số)', color='#00ffff', fontsize=11)
        ax2.set_xlabel('Tần số (Hz)', color='#888888')
        ax2.set_ylabel('Magnitude (dB)', color='#888888')
        ax2.set_xlim(0, sr//2)
        ax2.legend(fontsize=7, loc='upper right')
        ax2.grid(True, alpha=0.2)
        
        # --- Panel 3: Spectrogram ---
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.set_facecolor('#111111')
        S = librosa.stft(y_dsp, n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
        img = librosa.display.specshow(S_db, sr=sr, hop_length=config.HOP_LENGTH,
                                       x_axis='time', y_axis='hz', ax=ax3, cmap='magma')
        ax3.axhline(config.FILTER_LOW_FREQ, color='#00ffff', linestyle='--', alpha=0.5)
        ax3.axhline(config.FILTER_HIGH_FREQ, color='#00ffff', linestyle='--', alpha=0.5)
        ax3.set_title('Spectrogram (DSP)', color='#00ffff', fontsize=11)
        ax3.set_ylabel('Tần số (Hz)', color='#888888')
        fig.colorbar(img, ax=ax3, format='%+2.0f dB')
        
        # --- Panel 4: Mel Spectrogram ---
        ax4 = fig.add_subplot(gs[1, 1])
        ax4.set_facecolor('#111111')
        S_mel = librosa.feature.melspectrogram(y=y_dsp, sr=sr, n_mels=config.N_MELS,
                                               n_fft=config.N_FFT, hop_length=config.HOP_LENGTH,
                                               fmin=config.FMIN, fmax=config.FMAX)
        S_mel_db = librosa.power_to_db(S_mel, ref=np.max)
        img2 = librosa.display.specshow(S_mel_db, sr=sr, hop_length=config.HOP_LENGTH,
                                        x_axis='time', y_axis='mel', ax=ax4, cmap='magma',
                                        fmin=config.FMIN, fmax=config.FMAX)
        ax4.set_title('Mel Spectrogram (DSP) — Input cho CNN', color='#00ffff', fontsize=11)
        ax4.set_ylabel('Mel frequency', color='#888888')
        fig.colorbar(img2, ax=ax4, format='%+2.0f dB')
        
        # --- Panel 5: Energy per band (bar chart) ---
        ax5 = fig.add_subplot(gs[2, :])
        ax5.set_facecolor('#111111')
        
        bands = [
            ('0-100\nSub-bass', 0, 100),
            ('100-250\nBass', 100, 250),
            ('250-1k\nLow-mid', 250, 1000),
            ('1k-4k\nMid', 1000, 4000),
            ('4k-8k\nHigh', 4000, 8000),
            ('8k-11k\nV.High', 8000, 11025),
        ]
        
        total_energy = np.sum(fft_dsp**2)
        band_pcts = []
        for bname, lo, hi in bands:
            mask = (freqs >= lo) & (freqs < hi)
            pct = np.sum(fft_dsp[mask]**2) / total_energy * 100
            band_pcts.append(pct)
        
        colors = ['#a855f7', '#ff0080', '#00ff41', '#00ffff', '#ffaa00', '#ff4444']
        bars = ax5.bar([b[0] for b in bands], band_pcts, color=colors, edgecolor='#333333')
        
        # Add percentage labels
        for bar, pct in zip(bars, band_pcts):
            if pct > 1:
                ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'{pct:.1f}%', ha='center', color='#ffffff', fontsize=9)
        
        ax5.set_title(f'Phân bố năng lượng theo dải tần — Peak: {peak_freq:.0f} Hz',
                     color='#00ffff', fontsize=11)
        ax5.set_ylabel('% Năng lượng', color='#888888')
        ax5.grid(True, alpha=0.2, axis='y')
        
        plt.suptitle(f'{cls_name} — Mẫu {i+1}: {row["slice_file_name"]}',
                    color='#ffffff', fontsize=13, y=1.01)
        plt.savefig(f'{config.FIGURES_DIR}/fig_freq_analysis_{cls_name}_{i+1}.png',
                   dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
        plt.show()
    
    # ============ Summary for this class ============
    avg_fft = np.mean(all_fft_mag, axis=0)
    total = np.sum(avg_fft**2)
    
    print(f"\n{'─'*50}")
    print(f"  TÓM TẮT: {cls_name} ({vn_names[cls_name]})")
    print(f"{'─'*50}")
    
    peak_idx = np.argmax(avg_fft[1:]) + 1
    print(f"  Tần số đỉnh trung bình: {freqs[peak_idx]:.0f} Hz")
    print(f"  Phân bố năng lượng:")
    
    band_defs = [
        ('Sub-bass (0-100 Hz)', 0, 100),
        ('Bass (100-250 Hz)', 100, 250),
        ('Low-mid (250-1kHz)', 250, 1000),
        ('Mid (1k-4kHz)', 1000, 4000),
        ('High (4k-8kHz)', 4000, 8000),
        ('V.High (8k-11kHz)', 8000, 11025),
    ]
    
    for bname, lo, hi in band_defs:
        mask = (freqs >= lo) & (freqs < hi)
        pct = np.sum(avg_fft[mask]**2) / total * 100
        bar = '█' * int(pct / 2)
        print(f"    {bname:>25s}: {pct:5.1f}% {bar}")
    
    return avg_fft, freqs

## 2. Phân Tích Từng Class

### Class 0: air_conditioner (Máy điều hòa)
**Đặc điểm dự kiến**: Tiếng ù liên tục, tần số thấp, biên độ đều

In [ ]:
fft_ac, freqs = analyze_class('air_conditioner', n_samples=2)

### Class 1: car_horn (Còi xe hơi)
**Đặc điểm dự kiến**: Xung trung tần, ngắn, có thể lặp lại

In [ ]:
fft_ch, _ = analyze_class('car_horn', n_samples=2)

### Class 2: children_playing (Trẻ em chơi đùa)
**Đặc điểm dự kiến**: Phổ rộng, trung-cao tần, nhiều biến đổi

In [ ]:
fft_cp, _ = analyze_class('children_playing', n_samples=2)

### Class 3: dog_bark (Chó sủa)
**Đặc điểm dự kiến**: Xung trung tần, ngắn, lặp lại có nhịp

In [ ]:
fft_db, _ = analyze_class('dog_bark', n_samples=2)

### Class 4: drilling (Khoan đường)
**Đặc điểm dự kiến**: Phổ rộng, nhiều hài bậc cao, biên độ mạnh

In [ ]:
fft_dr, _ = analyze_class('drilling', n_samples=2)

### Class 5: engine_idling (Động cơ chạy không tải)
**Đặc điểm dự kiến**: Tần số rất thấp, ù đều, ít biến đổi

In [ ]:
fft_ei, _ = analyze_class('engine_idling', n_samples=2)

### Class 6: gun_shot (Tiếng súng)
**Đặc điểm dự kiến**: Xung cực ngắn, năng lượng cao, phổ rộng

In [ ]:
fft_gs, _ = analyze_class('gun_shot', n_samples=2)

### Class 7: jackhammer (Búa khoan)
**Đặc điểm dự kiến**: Va đập lặp nhanh, nhiều hài bậc, phổ rộng

In [ ]:
fft_jh, _ = analyze_class('jackhammer', n_samples=2)

### Class 8: siren (Còi báo động)
**Đặc điểm dự kiến**: Quét tần số lên xuống, trung-cao tần, tuần hoàn

In [ ]:
fft_si, _ = analyze_class('siren', n_samples=2)

### Class 9: street_music (Nhạc đường phố)
**Đặc điểm dự kiến**: Harmonic rõ, trung tần, có nhịp điệu

In [ ]:
fft_sm, _ = analyze_class('street_music', n_samples=2)

## 3. So Sánh Phổ Tần Số Giữa Tất Cả Classes

In [ ]:
# Analyze all classes with more samples for statistical robustness
np.random.seed(42)
n_stat_samples = 30

band_defs = [
    ('0-100', 0, 100),
    ('100-250', 100, 250),
    ('250-1k', 250, 1000),
    ('1k-4k', 1000, 4000),
    ('4k-8k', 4000, 8000),
    ('8k-11k', 8000, 11025),
]

class_band_data = {}

for cls_id, cls_name in enumerate(config.CLASS_NAMES):
    subset = metadata[metadata['classID'] == cls_id]
    sample_idx = np.random.choice(len(subset), min(n_stat_samples, len(subset)), replace=False)
    sample_rows = subset.iloc[sample_idx]
    
    band_pcts_list = []
    peak_freqs = []
    
    for _, row in sample_rows.iterrows():
        fpath = get_file_path(row)
        if not os.path.exists(fpath):
            continue
        y = load_audio(fpath)
        y_dsp = pipeline_b_process(y)
        
        fft = np.abs(np.fft.rfft(y_dsp))
        freqs = np.fft.rfftfreq(len(y_dsp), 1/sr)
        total = np.sum(fft**2)
        
        peak_freqs.append(freqs[np.argmax(fft[1:]) + 1])
        
        pcts = {}
        for bname, lo, hi in band_defs:
            mask = (freqs >= lo) & (freqs < hi)
            pcts[bname] = np.sum(fft[mask]**2) / total * 100
        band_pcts_list.append(pcts)
    
    avg_pcts = {}
    for bname, _, _ in band_defs:
        avg_pcts[bname] = np.mean([p[bname] for p in band_pcts_list])
    
    class_band_data[cls_name] = {
        'avg_pcts': avg_pcts,
        'peak_freq': np.mean(peak_freqs),
    }
    print(f"  Analyzed {cls_name}: peak={np.mean(peak_freqs):.0f} Hz")

print("\nDone!")

In [ ]:
# Stacked bar chart — energy distribution per class
fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')

x = np.arange(len(config.CLASS_NAMES))
width = 0.7
colors = ['#a855f7', '#ff0080', '#00ff41', '#00ffff', '#ffaa00', '#ff4444']
band_names = [b[0] for b in band_defs]

bottom = np.zeros(len(config.CLASS_NAMES))
for i, bname in enumerate(band_names):
    values = [class_band_data[c]['avg_pcts'][bname] for c in config.CLASS_NAMES]
    ax.bar(x, values, width, bottom=bottom, label=f'{bname} Hz',
           color=colors[i], edgecolor='#222222', linewidth=0.5)
    
    # Label significant bands
    for j, v in enumerate(values):
        if v > 15:
            ax.text(x[j], bottom[j] + v/2, f'{v:.0f}%',
                   ha='center', va='center', color='#000000', fontsize=7, fontweight='bold')
    bottom += values

# Add peak frequency markers
for j, cls_name in enumerate(config.CLASS_NAMES):
    peak = class_band_data[cls_name]['peak_freq']
    ax.text(x[j], 102, f'{peak:.0f} Hz', ha='center', color='#ffaa00', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([f'{c}\n(ID:{i})' for i, c in enumerate(config.CLASS_NAMES)],
                   fontsize=8, color='#ffffff')
ax.set_ylabel('% Năng lượng', color='#888888', fontsize=11)
ax.set_title('Phân Bố Năng Lượng Theo Dải Tần Số — Tất Cả Classes\n(Số trên đỉnh = tần số đỉnh trung bình)',
            color='#00ffff', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 115)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_frequency_distribution_all_classes.png',
           dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 4. Heatmap — Tương Đồng Tần Số Giữa Các Classes

In [ ]:
import seaborn as sns
from scipy.spatial.distance import cosine

# Build frequency profile matrix
profiles = []
for cls_name in config.CLASS_NAMES:
    pcts = class_band_data[cls_name]['avg_pcts']
    profiles.append([pcts[b[0]] for b in band_defs])

profiles = np.array(profiles)

# Compute cosine similarity
n = len(config.CLASS_NAMES)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = 1 - cosine(profiles[i], profiles[j])

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0a0a0a')

short_names = ['AC', 'Horn', 'Children', 'Dog', 'Drill',
               'Engine', 'Gun', 'Hammer', 'Siren', 'Music']

sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
           xticklabels=short_names, yticklabels=short_names,
           ax=ax, vmin=0.5, vmax=1.0, square=True,
           cbar_kws={'label': 'Cosine Similarity'})

ax.set_title('Độ Tương Đồng Tần Số Giữa Các Classes\n(Giá trị cao = phổ tần giống nhau → khó phân biệt)',
            color='#00ffff', fontsize=12)

plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_frequency_similarity_heatmap.png',
           dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

# Print most similar pairs
print("\nCác cặp class có phổ tần SỐ GIỐNG NHAU nhất (dễ nhầm lẫn):")
print("─" * 50)
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((sim_matrix[i,j], config.CLASS_NAMES[i], config.CLASS_NAMES[j]))
pairs.sort(reverse=True)
for sim, c1, c2 in pairs[:5]:
    print(f"  {c1:20s} ↔ {c2:20s} : similarity = {sim:.3f}")

print("\nCác cặp class có phổ tần SỐ KHÁC NHAU nhất (dễ phân biệt):")
print("─" * 50)
for sim, c1, c2 in pairs[-5:]:
    print(f"  {c1:20s} ↔ {c2:20s} : similarity = {sim:.3f}")

## 5. Kết Luận — Cách Xác Định Dải Tần Số

### Phương pháp xác định dải tần số cho mỗi class:

1. **Thu thập mẫu**: Lấy 30+ clip từ mỗi class
2. **FFT**: Chuyển từ miền thời gian → miền tần số
3. **Tính năng lượng theo band**: Chia phổ thành 6 dải, tính % năng lượng mỗi dải
4. **Xác định peak frequency**: Tần số có biên độ lớn nhất
5. **So sánh cross-class**: Dùng cosine similarity để tìm cặp dễ nhầm

### Tóm tắt đặc tính tần số:

| Class | Dải tần chính | Đặc điểm waveform |
|-------|--------------|--------------------|
| air_conditioner | 0–250 Hz | Liên tục, biên độ đều, ù |
| car_horn | 250–4k Hz | Xung ngắn, lặp lại |
| children_playing | 1k–4k Hz | Phổ rộng, nhiều biến đổi |
| dog_bark | 250–1k Hz | Xung ngắn, lặp có nhịp |
| drilling | 250–11k Hz | Phổ rộng nhất, nhiều hài bậc |
| engine_idling | 0–250 Hz | Liên tục, tần số rất thấp |
| gun_shot | 0–1k Hz | Xung cực mạnh, ngắn |
| jackhammer | 0–8k Hz | Va đập lặp nhanh |
| siren | 1k–4k Hz | Quét tần số lên xuống |
| street_music | 250–1k Hz | Harmonic, có nhịp điệu |